In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
def load_and_parse(file_path):
    """Load GH Archive data and parse nested fields"""
    events = []
    import gzip, json
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            events.append(json.loads(line))
    df = pd.DataFrame(events)

    # Parse nested fields
    df['actor_login'] = df['actor'].apply(lambda x: x.get('login') if isinstance(x, dict) else None)
    df['repo_name'] = df['repo'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

    return df

# Load two hours of data
df_12 = load_and_parse('raw_data/2025-01-01-12.json.gz')
df_13 = load_and_parse('raw_data/2025-01-01-13.json.gz')

print(f"Hour 12: {len(df_12)} events")
print(f"Hour 13: {len(df_13)} events")

Hour 12: 223540 events
Hour 13: 197224 events


AGGREGATE USER FEATURES


In [4]:
def aggregate_features(df, hour_label):
    """Aggregate user-level features from event data"""
    aggregated = df.groupby('actor_login').agg({
        'id': 'count',
        'type': lambda x: x.nunique(),
        'repo_name': lambda x: x.nunique(),
    }).rename(columns={
        'id': f'total_events_{hour_label}',
        'type': f'event_type_diversity_{hour_label}',
        'repo_name': f'unique_repos_{hour_label}'
    })
    return aggregated

# Create features (Hour 12)
X_raw = aggregate_features(df_12, 'h12')

# Create target (Hour 13 - active if > 3 events)
h13_activity = df_13.groupby('actor_login').agg({'id': 'count'}).rename(columns={'id': 'total_events_h13'})
active_threshold = h13_activity['total_events_h13'].quantile(0.75)
h13_activity['is_active_next_hour'] = (h13_activity['total_events_h13'] > active_threshold).astype(int)

# Merge features and target
data = X_raw.join(h13_activity[['is_active_next_hour']], how='inner')
print(f"Users active in both hours: {len(data)}")
print(f"\nColumns after merge: {data.columns.tolist()}")

print("\nFirst 5 rows of merged data:")
print(data.head())

Users active in both hours: 10124

Columns after merge: ['total_events_h12', 'event_type_diversity_h12', 'unique_repos_h12', 'is_active_next_hour']

First 5 rows of merged data:
              total_events_h12  event_type_diversity_h12  unique_repos_h12  \
actor_login                                                                  
0-y208                       3                         1                 2   
000Ljx                       1                         1                 1   
000deepak                    5                         1                 1   
00146664032q                 4                         1                 1   
001Rakib                     1                         1                 1   

              is_active_next_hour  
actor_login                        
0-y208                          0  
000Ljx                          0  
000deepak                       1  
00146664032q                    0  
001Rakib                        0  


HANDLE MISSING VALUES

In [5]:
print(data.isnull().sum())

# No missing values in our aggregated features
# (actor_login with NULL were automatically excluded by groupby)

# CAP OUTLIERS (WINSORIZATION)

def cap_outliers(df, column):
    """Cap values at 99th percentile"""
    cap = df[column].quantile(0.99)
    capped_count = (df[column] > cap).sum()
    df[f'{column}_capped'] = df[column].clip(upper=cap)
    print(f"  {column}: capped {capped_count} values at {cap:.1f}")
    return df, f'{column}_capped'

# Apply capping to skewed features
data, total_events_capped = cap_outliers(data, 'total_events_h12')
data, unique_repos_capped = cap_outliers(data, 'unique_repos_h12')

total_events_h12            0
event_type_diversity_h12    0
unique_repos_h12            0
is_active_next_hour         0
dtype: int64
  total_events_h12: capped 102 values at 66.8
  unique_repos_h12: capped 97 values at 6.0


CREATE DERIVED FEATURES

In [7]:
# 1. events_per_repo
data['events_per_repo'] = data['total_events_h12_capped'] / (data['unique_repos_h12_capped'] + 1)

# 2. diversity_score
data['diversity_score'] = data['event_type_diversity_h12'] / data['total_events_h12_capped'].clip(lower=1)

# 3. log_total_events
data['log_total_events'] = np.log1p(data['total_events_h12_capped'])

# 4. is_bot_suspect
data['is_bot_suspect'] = (data['total_events_h12_capped'] > 1000).astype(int)
bot_count = data['is_bot_suspect'].sum()


SELECT FINAL FEATURES

In [8]:
# Features to use
feature_columns = [
    'total_events_h12_capped',
    'event_type_diversity_h12',
    'unique_repos_h12_capped',
    'events_per_repo',
    'diversity_score',
    'log_total_events',
    'is_bot_suspect'
]

X = data[feature_columns].copy()
y = data['is_active_next_hour'].copy()

print(f"Selected features: {feature_columns}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:\n{y.value_counts()}")

Selected features: ['total_events_h12_capped', 'event_type_diversity_h12', 'unique_repos_h12_capped', 'events_per_repo', 'diversity_score', 'log_total_events', 'is_bot_suspect']
X shape: (10124, 7)
y shape: (10124,)

Class distribution:
is_active_next_hour
0    7141
1    2983
Name: count, dtype: int64


NORMALIZATION

In [9]:
# Check skew before normalization
print("Skew before normalization:")
for col in X.columns:
    skew = X[col].skew()
    print(f"  {col}: {skew:.2f}")

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("\nStandardScaler applied (mean=0, std=1 for all features)")

# Check after normalization
print("\nStatistics after normalization:")
print(X_scaled.describe().round(2))

Skew before normalization:
  total_events_h12_capped: 5.43
  event_type_diversity_h12: 2.88
  unique_repos_h12_capped: 4.43
  events_per_repo: 6.29
  diversity_score: -0.20
  log_total_events: 1.72
  is_bot_suspect: 0.00

StandardScaler applied (mean=0, std=1 for all features)

Statistics after normalization:
       total_events_h12_capped  event_type_diversity_h12  \
count                 10124.00                  10124.00   
mean                      0.00                      0.00   
std                       1.00                      1.00   
min                      -0.39                     -0.45   
25%                      -0.39                     -0.45   
50%                      -0.27                     -0.45   
75%                      -0.04                     -0.45   
max                       7.17                     10.80   

       unique_repos_h12_capped  events_per_repo  diversity_score  \
count                 10124.00         10124.00         10124.00   
mean        

SPLIT AND SAVE PREPROCESSED DATA

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

# Save datasets
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False, header=True)
y_test.to_csv('y_test.csv', index=False, header=True)
data.to_csv('full_preprocessed_data.csv', index=False)

Train size: 8099 (80.0%)
Test size: 2025 (20.0%)

Train class distribution:
is_active_next_hour
0    5713
1    2386
Name: count, dtype: int64

Test class distribution:
is_active_next_hour
0    1428
1     597
Name: count, dtype: int64


No missing values were found. Outliers were capped at the 99th percentile - total events capped at 66.8 events per hour, unique repositories capped at 6 repos.

Four derived features were created including log transformation which reduced skewness from 5.43 to 1.72. StandardScaler was applied successfully.

The data was split 80-20 with class distribution preserved - 70.5 percent inactive, 29.5 percent active. All files are saved and ready for model training.

## Feature Description for ML Developer

**Features (X):**

- total_events_h12_capped - Number of user events in Hour 12, capped at 99th percentile (max 66.8). Primary activity indicator.

- event_type_diversity_h12 - Number of unique event types performed by user in Hour 12 (range 1 to 10). Shows engagement breadth.

- unique_repos_h12_capped - Number of unique repositories user interacted with in Hour 12, capped at 99th percentile (max 6). Shows GitHub reach.

- events_per_repo - Total events divided by unique repos. Measures activity intensity vs breadth. Higher values mean deeper engagement with fewer repos.

- diversity_score - Event type diversity divided by total events. High score with low total events indicates engaged new user.

- log_total_events - Log transformation of capped total events. Reduces skew for linear models.

- is_bot_suspect - Binary flag for total events above 1000. Currently all zeros due to capping. Can be removed.

**Target (y):**

- is_active_next_hour - Binary classification (1 = active, 0 = inactive). User is active in Hour 13 if they had more than 3 events (75th percentile threshold).